**Importing Libraries**

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

**T-net Architecture**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
!git clone https://github.com/fxia22/pointnet.pytorch
%cd pointnet.pytorch
!pip install -e .

Cloning into 'pointnet.pytorch'...
remote: Enumerating objects: 213, done.
remote: Counting objects: 100% (2/2), done.
remote: Compressing objects: 100% (2/2), done.
remote: Total 213 (delta 0), reused 2 (delta 0), pack-reused 211
Receiving objects: 100% (213/213), 229.91 KiB | 972.00 KiB/s, done.
Resolving deltas: 100% (125/125), done.
/content/pointnet.pytorch
Obtaining file:///content/pointnet.pytorch
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 53.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.6/823.6 kB 49.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 68.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 731.7/731.7 MB 2.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.6/410.6 MB 4.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 MB 8.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 MB 1

In [ ]:
class Tnet(nn.Module):
  def __init__(self, dim, n_points):
    super(Tnet,self).__init__()

    self.dim=dim

    self.conv1=nn.Conv1d(dim,64,kernel_size=1)
    self.conv2=nn.Conv1d(64,128,kernel_size=1)
    self.conv3=nn.Conv1d(128,1024,kernel_size=1)

    self.linear1=nn.Linear(1024,512)
    self.linear2=nn.Linear(512,256)
    self.linear3=nn.Linear(256,dim**2)

    self.bn1=nn.BatchNorm1d(64)
    self.bn2=nn.BatchNorm1d(128)
    self.bn3=nn.BatchNorm1d(1024)
    self.bn4=nn.BatchNorm1d(512)
    self.bn5=nn.BatchNorm1d(256)

    self.max_pool= nn.MaxPool1d(kernel_size=n_points)


  def forward(self,x):
    bs=x.shape[0]


    x=self.bn1(F.relu(self.conv1(x)))
    x=self.bn2(F.relu(self.conv2(x)))
    x=self.bn3(F.relu(self.conv3(x)))

    x=self.max_pool(x).view(bs,-1)

    x=self.bn4(F.relu(self.linear1(x)))
    x=self.bn5(F.relu(self.linear2(x)))
    x=self.linear3(x)


    return x


In [ ]:
v=Tnet(dim=3,n_points=2500)
input=torch.randn(32,3,2500)
r=v(input)
print(r.shape)

torch.Size([32, 9])


**PointNet Architecture:**

In [ ]:
class PointNetBody(nn.Module):

  def __init__(self, n_points, n_global_feats=1024, local_feats=True):

    super(PointNetBody,self).__init__()

    self.n_points=n_points
    self.n_global_feats=n_global_feats
    self.local_feats=local_feats

    self.tnet1=Tnet(dim=3,n_points=n_points)   #input transform
    self.tnet2=Tnet(dim=64,n_points=n_points)  #feature transform

    #mlp(64,64)
    self.conv1=nn.Conv1d(3,64,kernerl_size=1)
    self.conv1=nn.Conv1d(64,64,kernerl_size=1)

    #mlp(64,128,1024)
    self.conv3=nn.Conv1d(64,64,kernel_size=1)
    self.conv4=nn.Conv1d(64,128,kernel_size=1)
    self.conv5=nn.Conv1d(128,1024,kernel_size=1)

    #batch normalization
    self.bn1=nn.BatchNorm1d(64)
    self.bn2=nn.BatchNorm1d(64)
    self.bn3=nn.BatchNorm1d(64)
    self.bn4=nn.BatchNorm1d(128)
    self.bn5=nn.BatchNorm1d(1024)

    #max pooling
    self.max_pool=nn.MaxPool1d(kernel_size=n_points,return_indices=True)



  def forward(self,x):
    batch_size=x.shape[0]

    #input transform matrix
    A_input=self.tnet1(x)
    x=torch.bmm(x.transpose(2,1),A_input).transpose(2,1)

    x=self.bn1(F.relu(self.conv1(x)))
    x=self.bn2(F.relu(self.conv2(x)))

    #feature transform matrix
    A_feat=self.tnet2(x)
    x=torch.bmm(x.transpose(2,1),A_feat).transpose(2,1)

    local_features=x.clone()

    x=self.bn3(F.relu(self.conv3(x)))
    x=self.bn4(F.relu(self.conv4(x)))
    x=self.bn5(F.relu(self.conv5(x)))

    global_features,critical_indicies=self.max_pool(x)
    global_features=global_features.view(batch_size,-1)
    critical_indicies=critical_indicies.view(batch_size,-1)


    if self.local_feats:
      features=torch.cat((local_features,global_features.unsqueeze(-1).repeat(1,1,self.n_points)),dim=1)
      return features,critical_indicies, A_feat

    else:
      return global_features,critical_indicies, A_feat


In [ ]:
class PointNetClassification(nn.Module):
  def __init__(self, n_points, n_global_feats=1024,k=2):
    super(PointNetClassification,self).__init__()
    self.body=PointNetBody(n_points,n_global_feats,local_feats=False)

    #mlp(512,256,k)
    self.linear1=nn.Linear(n_global_feats,512)
    self.linear2=nn.Linear(512,256)
    self.linear3=nn.Linear(256,k)

    bn1=nn.BatchNorm1d(512)
    bn2=nn.BatchNorm1d(256)

    self.dropout=nn.Dropout(p=0.3)


  def forward(self,x):
    x,critical_indicies,A_feat=self.body(x)

    x=self.bn1(F.relu(self.linear1(x)))
    x=self.bn2(F.relu(self.linear2(x)))

    x=self.dropout(x)
    x=self.linear3(x)

    return x,critical_indicies, A_feat


In [ ]:
class PointNetSegmentation(nn.Module):
  def __init__(self, n_points, n_global_feats=1024,m=2):
    super(PointNetSegmentation,self).__init__()
    self.body=PointNetBody(n_points,n_global_feats,local_feats=True)

    self.n_points=n_points
    self.m=m

    #mlp(512,256,128) and mlp(128,m)
    ttl_features=n_global_feats+ 64
    self.conv1=nn.Conv1d(ttl_features,512)
    self.conv2=nn.Conv1d(512,256)
    self.conv3=nn.Conv1d(256,128)
    self.conv4=nn.Conv1d(128,m)

    self.bn1=nn.BatchNorm1d(512)
    self.bn2=nn.BatchNorm1d(256)
    self.bn3=nn.BatchNorm1d(128)


  def forward(self,x):
    x,critical_indicies,A_feat=self.body(x)


    x=self.bn1(F.relu(self.conv1(x)))
    x=self.bn2(F.relu(self.conv2(x)))
    x=self.bn3(F.relu(self.conv3(x)))
    x=self.conv4(x)
    x=x.transpose(2,1)

    return x,critical_indicies, A_feat



**Loss Functions**

In [ ]:
def PointNetLoss(predictions,target,A=None, alpha=None, gamma=0
                 ,reg_weight=0,size_average=True):
  batch_size=predictions.size(0)
  cross_entropy=nn.CrossEntropyLoss(weight=alpha)
  ce_loss=cross_entropy(predictions,targets)
  pn=F.softmax(predictions)
  pn=pn.gather(1,targets.view(-1,1)).view(-1)

  if reg_weight>0:
    I=torch.eye(64).unsqueeze(0).repeat(A.shape[0],1,1)
    reg=torch.linalg.norm(I-torch.bmm(A,A.transpose(2,1)))
    reg=reg_weight*reg/batch_size
  else:
    reg=0

  loss=((1-pn)**gamma*ce_loss)

  if size_average:
    return loss.mean()+reg
  else:
    return loss.sum()+reg

def PointNetSegLoss(predictions, targets, pred_choice=None,alpha=None, gamma=0, size_average=True, dice=False):
  if isinstance(alpha, (float, int)):
      alpha = torch.Tensor([alpha, 1 - alpha])
  if isinstance(alpha, (list, np.ndarray)):
      alpha = torch.Tensor(alpha)

  cross_entropy=nn.CrossEntropyLoss(weight=alpha)
  ce_loss=cross_entropy(predictions.transpose(2,1),targets)
  predictions=predictions.contiguous().view(-1,predictions.size(2))

  pn=F.softmax(predictions)
  pn=pn.gather(1,targets.view(-1,1)).view(-1)

  loss=((1-pn)**gamma*ce_loss)
  if size_average:
    loss=loss.mean()
  else:
    loss=loss.sum()

  if dice:
    return loss+dice_loss(targets,pred_choice,eps=1)
  else:
    return loss

def dice_loss(predictions, targets, eps=1):
    targets = targets.reshape(-1)
    predictions = predictions.reshape(-1)

    cats = torch.unique(targets)

    top = 0
    bot = 0
    for c in cats:
        locs = targets == c

        y_tru = targets[locs]
        y_hat = predictions[locs]

        top += torch.sum(y_hat == y_tru)
        bot += len(y_tru) + len(y_hat)

    return 1 - 2 * ((top + eps) / (bot + eps))



In [ ]:
!unzip /content/drive/MyDrive/Dataset/shapenetcore_partanno_segmentation_benchmark_v0.zip -d /content


Streaming output truncated to the last 5000 lines.
  inflating: /content/shapenetcore_partanno_segmentation_benchmark_v0/02691156/seg_img/e1225308d6c26c862b600da24e0965.png  
  inflating: /content/shapenetcore_partanno_segmentation_benchmark_v0/02691156/seg_img/46ae88cad17edca7ae7c0d0e12bd33da.png  
  inflating: /content/shapenetcore_partanno_segmentation_benchmark_v0/02691156/seg_img/1ea7a36e4f353416fe1f6e05091d5d9.png  
  inflating: /content/shapenetcore_partanno_segmentation_benchmark_v0/02691156/seg_img/e94ad5f8e53a255a8fc2d09ac4aa4e78.png  
  inflating: /content/shapenetcore_partanno_segmentation_benchmark_v0/02691156/seg_img/8b61ba80d9e487deca8607f540cc62ba.png  
  inflating: /content/shapenetcore_partanno_segmentation_benchmark_v0/02691156/seg_img/3cbc83ba49edeccebc0909d98a1ff2b4.png  
  inflating: /content/shapenetcore_partanno_segmentation_benchmark_v0/02691156/seg_img/c5d0dd7a7b44b079a76ffc04f04676cb.png  
  inflating: /content/shapenetcore_partanno_segmentation_benchmark_v0/

In [ ]:
!pip install plyfile


In [ ]:
datapath = '/content/shapenetcore'

!python /content/dataset.py ShapeNetDataset datapath

In [ ]:
!python /content/pointnet.pytorch/utils/train_classification.py --dataset /content/shapenetcore_partanno_segmentation_benchmark_v0 --nepoch=11 --dataset_type shapenet


Namespace(batchSize=32, num_points=2500, workers=4, nepoch=11, outf='cls', model='', dataset='/content/shapenetcore_partanno_segmentation_benchmark_v0', dataset_type='shapenet', feature_transform=False)
Random Seed:  260
{'Airplane': 0, 'Bag': 1, 'Cap': 2, 'Car': 3, 'Chair': 4, 'Earphone': 5, 'Guitar': 6, 'Knife': 7, 'Lamp': 8, 'Laptop': 9, 'Motorbike': 10, 'Mug': 11, 'Pistol': 12, 'Rocket': 13, 'Skateboard': 14, 'Table': 15}
{'Airplane': 4, 'Bag': 2, 'Cap': 2, 'Car': 4, 'Chair': 4, 'Earphone': 3, 'Guitar': 3, 'Knife': 2, 'Lamp': 4, 'Laptop': 2, 'Motorbike': 6, 'Mug': 2, 'Pistol': 3, 'Rocket': 3, 'Skateboard': 3, 'Table': 3} 4
{'Airplane': 0, 'Bag': 1, 'Cap': 2, 'Car': 3, 'Chair': 4, 'Earphone': 5, 'Guitar': 6, 'Knife': 7, 'Lamp': 8, 'Laptop': 9, 'Motorbike': 10, 'Mug': 11, 'Pistol': 12, 'Rocket': 13, 'Skateboard': 14, 'Table': 15}
{'Airplane': 4, 'Bag': 2, 'Cap': 2, 'Car': 4, 'Chair': 4, 'Earphone': 3, 'Guitar': 3, 'Knife': 2, 'Lamp': 4, 'Laptop': 2, 'Motorbike': 6, 'Mug': 2, 'Pistol'

In [ ]:
!python /content/pointnet.pytorch/utils/show_cls.py --model  /content/pointnet.pytorch/cls/cls_model_10.pth

Namespace(model='/content/pointnet.pytorch/cls/cls_model_10.pth', num_points=2500)
Traceback (most recent call last):
  File "/content/pointnet.pytorch/utils/show_cls.py", line 23, in <module>
    test_dataset =ShapeNetDataset(
  File "/content/pointnet.pytorch/pointnet/dataset.py", line 72, in __init__
    with open(self.catfile, 'r') as f:
FileNotFoundError: [Errno 2] No such file or directory: 'shapenetcore_partanno_segmentation_benchmark_v0/synsetoffset2category.txt'
